# Chaton — Colab training notebook

Trains Chaton on a Colab T4 (16GB) — much bigger than the local 8GB RTX 2070.

Flow:
1. Mount Google Drive (so model.pt survives after the session dies).
2. git clone your repo into /content/le-gros-chaton (set REPO below).
3. Install deps, set GPU overrides (bigger batch, wikitext-103 corpus).
4. Build the data memmap (one-time; downloads wikitext-103 ~500MB).
5. Train. Loss prints every 250 steps. ~1-3h on a T4.
6. Save model.pt to Drive.
7. (optional) generate a sample.

**Before running:** set REPO to your GitHub URL, and Run Runtime → Change runtime type → T4 GPU.

In [ ]:
# 1. Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
assert torch.cuda.is_available(), 'Runtime → Change runtime type → T4 GPU'

In [ ]:
# 2. Mount Google Drive (weights saved here survive session end)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Get the code onto the VM. SET THIS to your GitHub repo URL.
#    If private, Colab may prompt for a token — or make it public for simplicity.
REPO = 'https://github.com/Mateooo93/le-gros-chaton.git'
%cd /content
!rm -rf le-gros-chaton
!git clone {REPO} le-gros-chaton
%cd /content/le-gros-chaton
!ls -la

In [ ]:
# 4. Install deps — PIN datasets + huggingface_hub to versions compatible with
#    wikitext (newer huggingface_hub breaks on wikitext's old pinned revision URI).
!pip install -q tiktoken tokenizers "datasets==2.21.0" "huggingface_hub<0.30"
# After installing, RESTART the runtime (Runtime → Restart session) so the
# pinned versions actually take effect, then continue from cell 5.

In [ ]:
# 5. GPU overrides. T4 16GB -> micro_batch 32 x accum 2 = effective batch 64.
import os
os.environ['CHATON_CORPUS']      = 'wikitext-103'   # ~100M tokens
os.environ['CHATON_MICRO_BATCH'] = '32'
os.environ['CHATON_GRAD_ACCUM']  = '2'              # effective batch = 64
os.environ['CHATON_MAX_ITERS']   = '12000'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('overrides set; effective batch =', 32*2)

In [ ]:
# 6. Build the data memmap (downloads wikitext-103 ~500MB, encodes ~200MB). One-time per VM.
!python data2.py

In [ ]:
# 7. Sanity check: model builds + start loss ~ ln(vocab)=10.8 (init fix working, not 158)
!python model.py

In [ ]:
# 8. TRAIN — the long cell. wikitext-103, 12k steps on a T4 ≈ 1-3 hours.
#    Keep this tab open; Colab disconnects after ~90 min idle.
!python -u train.py

In [ ]:
# 9. Save model.pt (and the memmaps, optional) to Drive so they survive.
import shutil, os
DRIVE_DIR = '/content/drive/MyDrive/chaton'
os.makedirs(DRIVE_DIR, exist_ok=True)
import glob
memmaps = glob.glob('train_tokens_*.bin') + glob.glob('val_tokens_*.bin')
for f in ['model.pt'] + memmaps:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(DRIVE_DIR, f))
        print('copied', f, '->', DRIVE_DIR)
print('done — model.pt is in your Drive at', DRIVE_DIR)

In [ ]:
# 10. Optional: sample from the freshly trained base model.
from tokenizer import decode, encode, VOCAB_SIZE
import torch
from model import GPT
device = torch.device('cuda')
m = GPT().to(device)
m.load_state_dict(torch.load('model.pt', map_location=device))
m.eval()
prompt = 'The history of Singapore'
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
out = m.generate(idx, max_new_tokens=120, temperature=0.8, top_k=50)
print(prompt + decode(out[0].tolist()[len(idx[0]):]))

## After training

- **Base model done.** Download `model.pt` from Drive (or Drive desktop app) back to `~/le-gros-chaton` to fine-tune locally on the 2070: `python finetune.py` then `python chat_finetuned.py`.
- **Capstone (optional, big win):** set `vocab_size=16384` + `n_embd=384, n_layer=8, n_head=6` in config.py, run `CHATON_CUSTOM_BPE=1 python train_custom_tokenizer.py` to train the 16k BPE, push to git, then retrain from scratch here. Full retrain — do it on Colab.
- Old `model.pt`/`model_finetuned.pt` from before the redesign are architecture-incompatible (no wpe, RoPE, RMSNorm) — overwritten by this run.

## To re-run in a new session
Re-mount Drive, re-clone (cell 3), re-run 1-8. The memmap rebuilds on the fresh VM (cell 6). To avoid retraining from scratch, you'd need checkpoint-resume code — not implemented yet, so plan to finish a run in one session.